In [ ]:
import sys
import yaml
import xarray as xr
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
from dash import Dash, html, dcc, Input, Output, State, ctx, no_update
from toolbox.pipeline import Pipeline, _setup_logging

# --- UI Configuration Variables ---
UI_MAX_WIDTH = "600px"
UI_COMPACT_PADDING = "6px 10px"
UI_BORDER_RADIUS = "4px"
UI_MAIN_COLOUR = "#2c3e50"
UI_BUTTON_COLOUR = "#3498db"
UI_BUTTON_HOVER = "#2980b9"
UI_BG_COLOUR = "#f8f9fa"

app = Dash(__name__)

In [ ]:
CHLA_DEPTH_RATIO = 0.9 
DEFAULT_CHLA_DEPTH = 550

BASE_CONFIG_YAML = """
pipeline:
  name: Example CTD Processing Pipeline
  description: A pipeline for processing CTD data
  visualisation: false

steps:
  - name: Load OG1
    parameters:
      file_path: PLACEHOLDER
    diagnostics: false

  - name: Apply QC
    parameters:
      qc_settings:
        impossible date qc: {}
        impossible location qc: {}
        position on land qc: {}
        impossible speed qc: {}
    diagnostics: false

  - name: Apply QC
    parameters:
      qc_settings:
        impossible range qc:
          variable_ranges:
            PRES:
              3: [-5, -2.4]
              4: [-.inf, -5]
          also_flag:
            PRES: [CNDC, TEMP]
          plot: [PRES]
        stuck value qc:
          variables:
            PRES: 2
          also_flag:
            PRES: [CNDC, TEMP]
          plot: [PRES]
    diagnostics: false

  - name: Interpolate Data
    parameters:
      qc_handling_settings:
        flag_filter_settings:
          PRES: [3, 4, 9]
          LATITUDE: [3, 4, 9]
          LONGITUDE: [3, 4, 9]
        reconstruction_behaviour: replace
        flag_mapping:
          3: 8
          4: 8
          9: 8
    diagnostics: false

  - name: Derive CTD
    parameters:
      to_derive: [DEPTH]
    diagnostics: false

  - name: Find Profiles Beta
    diagnostics: false

  - name: Apply QC
    parameters:
      qc_settings:
          valid profile qc:
            profile_length: 50
            depth_range: [-1000, 0]
    diagnostics: false

  - name: Salinity Adjustment
    parameters:
      qc_handling_settings:
        flag_filter_settings:
          CNDC: [3, 4, 9]
          TEMP: [3, 4, 9]
          PROFILE_NUMBER: [3, 4, 9]
        reconstruction_behaviour: reinsert
        flag_mapping:
          0: 5
          1: 5
          2: 5
      filter_window_size: 21
      plot_profiles_in_range: [100, 150]
    diagnostics: false

  - name: Derive CTD
    parameters:
      to_derive: [PRAC_SALINITY, ABS_SALINITY, CONS_TEMP, DENSITY]
    diagnostics: false

  - name: Chla Deep Correction
    parameters:
      apply_to: CHLA
      dark_value: null
      depth_threshold: -550
    diagnostics: false

  - name: Chla Quenching Correction
    parameters:
      method: Argo
      apply_to: CHLA
      mld_settings:
        threshold_on: DENSITY
        reference_depth: -10
        threshold: 0.05
      plot_profiles: [101, 200, 201, 300, 301, 400]
    diagnostics: false

  - name: BBP from Beta
    parameters:
      theta: 124
      xfactor: 1.076
    diagnostics: false

  - name: Isolate BBP Spikes
    parameters:
      window_size: 50
      method: median
    diagnostics: false

  - name: Data Export
    parameters:
      export_format: netcdf
      output_path: PLACEHOLDER
"""

def generate_file_config(input_path: Path, output_path: Path, config_path: Path, chla_ratio: float, chla_depth: int) -> None:
    config = yaml.safe_load(BASE_CONFIG_YAML)
    config["steps"][0]["parameters"]["file_path"] = str(input_path)
    config["steps"][-1]["parameters"]["output_path"] = str(output_path)
    
    with xr.open_dataset(input_path) as ds:
        has_location = "LATITUDE" in ds.variables and "LONGITUDE" in ds.variables
        
        max_depth = 1000 
        if "DEPTH" in ds.variables:
            max_depth = float(ds["DEPTH"].max())
        elif "PRES" in ds.variables:
            max_depth = float(ds["PRES"].max())

    if not has_location:
        qc_settings = config["steps"][1]["parameters"]["qc_settings"]
        for test in ["impossible location qc", "position on land qc", "impossible speed qc"]:
            qc_settings.pop(test, None)
            
    calculated_threshold = -int(max_depth * chla_ratio)
    final_threshold = max(-chla_depth, calculated_threshold)
    
    for step in config["steps"]:
        if step["name"] == "Chla Deep Correction":
            step["parameters"]["depth_threshold"] = final_threshold

    with open(config_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)

def run_pipeline(config_path: Path) -> str:
    try:
        with open(config_path, 'r') as file:
            config = yaml.safe_load(file)
            
        p = Pipeline()
        p.global_parameters = config.get("pipeline", {})
        p.logger = _setup_logging() 
        p.build_steps(config.get("steps", []))
        
        p.run()
        return "Success"
        
    except FileNotFoundError:
        return f"Failed: Could not find config at {config_path.name}"
    except yaml.YAMLError as exc:
        return f"Failed: Error parsing YAML file. {exc}"
    except Exception as e:
        return f"Failed: {e}"

In [ ]:
# --- Styling Dictionaries ---
container_style = {
    "maxWidth": UI_MAX_WIDTH,
    "margin": "0 auto",
    "padding": "20px",
    "fontFamily": "sans-serif",
    "backgroundColor": UI_BG_COLOUR,
    "borderRadius": UI_BORDER_RADIUS,
    "border": "1px solid #e0e0e0"
}

input_style = {
    "width": "100%",
    "padding": UI_COMPACT_PADDING,
    "marginBottom": "12px",
    "borderRadius": UI_BORDER_RADIUS,
    "border": "1px solid #ccc",
    "boxSizing": "border-box"
}

label_style = {
    "display": "block",
    "marginBottom": "4px",
    "fontWeight": "bold",
    "color": UI_MAIN_COLOUR,
    "fontSize": "14px"
}

button_style = {
    "width": "100%",
    "padding": "10px",
    "backgroundColor": UI_BUTTON_COLOUR,
    "color": "white",
    "border": "none",
    "borderRadius": UI_BORDER_RADIUS,
    "cursor": "pointer",
    "fontWeight": "bold",
    "marginTop": "10px",
    "marginBottom": "20px"
}

log_area_style = {
    "width": "100%",
    "height": "200px",
    "padding": UI_COMPACT_PADDING,
    "borderRadius": UI_BORDER_RADIUS,
    "border": "1px solid #ccc",
    "backgroundColor": "#fff",
    "fontFamily": "monospace",
    "fontSize": "12px",
    "whiteSpace": "pre-wrap",
    "overflowY": "auto",
    "boxSizing": "border-box"
}

# --- Dash Layout ---
app.layout = html.Div(style=container_style, children=[
    html.H3("CTD Pipeline Batch Processor", style={"color": UI_MAIN_COLOUR, "marginTop": "0"}),
    
    html.Label("Input Directory:", style=label_style),
    dcc.Input(id="input-dir", type="text", value="/Users/orlpru/Desktop/OG1_Data/test_data", style=input_style, debounce=False),
    
    html.Label("Output Directory:", style=label_style),
    dcc.Input(id="output-dir", type="text", value="/Users/orlpru/Desktop/OG1_Data/test_output", style=input_style, debounce=False),
    
    html.Label("CHLA Depth Ratio:", style=label_style),
    dcc.Slider(id="chla-ratio", min=0.1, max=1.0, step=0.05, value=0.9, 
               marks={i/10: str(i/10) for i in range(1, 11)}),
    
    html.Div(style={"marginTop": "15px"}),
    
    html.Label("Maximum CHLA Depth (m):", style=label_style),
    dcc.Input(id="chla-depth", type="number", value=550, style=input_style, debounce=False),
    
    html.Button("Run Pipeline", id="run-button", n_clicks=0, style=button_style),
    
    html.Label("Processing Output:", style=label_style),
    html.Div(id="output-log", style=log_area_style, children="Ready.")
])

# --- Callbacks ---
@app.callback(
    Output("output-log", "children"),
    Input("run-button", "n_clicks"),
    Input("input-dir", "n_submit"),
    Input("output-dir", "n_submit"),
    Input("chla-depth", "n_submit"),
    State("input-dir", "value"),
    State("output-dir", "value"),
    State("chla-ratio", "value"),
    State("chla-depth", "value"),
    prevent_initial_call=True
)
def process_batch(n_clicks, in_submit, out_submit, depth_submit, in_dir_str, out_dir_str, chla_ratio, chla_depth):
    trigger = ctx.triggered_id
    if not trigger:
        return no_update

    in_dir = Path(in_dir_str)
    out_dir = Path(out_dir_str)
    
    if not in_dir.exists():
        return f"Error: Input directory does not exist.\n{in_dir}"
        
    out_dir.mkdir(parents=True, exist_ok=True)
    
    files = list(in_dir.glob("*.nc"))
    if not files:
        return "No .nc files found in the specified input directory."
        
    successful = []
    failed = []
    log_messages = []
    
    log_messages.append(f"Starting batch process for {len(files)} files...\n")
    
    for input_file in files:
        file_stem = input_file.stem
        output_file = out_dir / f"{file_stem}_Processed.nc"
        config_file = out_dir / f"{file_stem}_config.yaml"
        log_file = out_dir / f"{file_stem}_log.txt"
        
        generate_file_config(input_file, output_file, config_file, chla_ratio, chla_depth)
        
        with open(log_file, 'w') as f:
            with redirect_stdout(f), redirect_stderr(f):
                status = run_pipeline(config_file)
        
        if status == "Success":
            successful.append(input_file.name)
            log_messages.append(f"[OK] {input_file.name}")
        else:
            failed.append((input_file.name, status))
            log_messages.append(f"[FAILED] {input_file.name}: {status}")
            
    summary = "\n--- Batch Processing Summary ---\n"
    summary += f"Total processed: {len(successful) + len(failed)}\n"
    summary += f"Successful: {len(successful)}\n"
    summary += f"Failed: {len(failed)}\n"
    
    for filename, error_msg in failed:
        summary += f"  * {filename}: {error_msg}\n"
        
    log_messages.append(summary)
    
    return "\n".join(log_messages)

# Run the app inline within the Jupyter Notebook
if __name__ == '__main__':
    app.run(jupyter_mode="inline")